it is also called as the thread memory

The history for the frontend is managed in seperate backed. In thread memory we implement this for proper AI response maintaing context.

We properly store the context by using n previous messages and history of all other messages older than last n messages

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
import operator
from langchain_openai import ChatOpenAI
from langchain_core.messages import trim_messages, BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import RemoveMessage
from langchain_core.messages.utils import count_tokens_approximately

In [ ]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",  # Your Azure deployment name
    base_url="https://openai-rg-nadeem.openai.azure.com/openai/v1",
    #add your api key below
    # api_key=
    )

MAX_TOKENS = 150

In [ ]:
class State(TypedDict):
    Human: str
    AI: str
    summary: str
    messages: Annotated[list[BaseMessage], operator.add]

In [ ]:
def chat_node(state: State):
    messages= []  
    trimmed_messags= trim_messages(
    messages=state["messages"],
    max_tokens=MAX_TOKENS,
    strategy="last",
    token_counter=count_tokens_approximately,
)
    messages.append(trimmed_messags)
    for message in messages:
        print(message)
    print("Tokens count", count_tokens_approximately(messages=trim_messages))
    response = llm.invoke(messages)
    return {
    "messages": response
    }

    

In [ ]:
def summary_node(state : State):
    messages = state['messages']
    summary = state['summary']
    if summary:
        prompt = (
            f"Extend the summary {state['summary']} using the new messages: {state['messages']}"
        )
    else:
        prompt=(
            f"Generate summary of these messages {state['messages']}"
        )
    messages += HumanMessage(content=prompt)
    response = llm.invoke(messages)

    messages_to_delete = [
        RemoveMessage(id=msg.id)
        for msg in messages[:4]
        ]
    
    return {
        "messages":messages_to_delete,
        "summary": response.content
    }

def need_summary(state:State):
    return len(state['messages'])>6



    

In [ ]:

graph = StateGraph(State)
graph.add_node('chat', chat_node)
graph.add_node('summary', summary_node)
graph.add_edge(START, 'chat')
graph.add_conditional_edges('chat', need_summary, 
                            {
                                True:'summary',
                                False:END
                            }
)
graph.add_edge('summary', END)
memory_saver = MemorySaver()
workflow = graph.build(checkpointer=memory_saver)

config = {
    "configurable":{
        "thread_id":"nadeem"
    }
}



In [ ]:
initial_state = {
    "Human": HumanMessage('What is this are you ohkk'),
    "messages": []
}
execution = workflow.invoke(initial_state)
print(execution)